# Fase 3 · Representación VSM binaria (scikit-learn)

Este notebook construye las representaciones **binarias** del corpus normalizado (`data/scientific_normalized_corpus.tsv`) con `CountVectorizer(binary=True)` de scikit-learn.

**Objetivo:** generar las 4 configuraciones binarias del plan, {título, abstract} × {unigrama, bigrama}, y guardar cada vectorizador con su matriz en `models/{campo}_{ngrama}_binary.pkl`, con la misma estructura que los `.pkl` de TF-IDF (notebook 03).

## ¿Qué es la representación binaria?

Es el modelo vectorial (VSM) más simple. Cada documento $d_i$ es un vector con una posición por término $t_j$ del vocabulario, y el valor solo dice **si el término aparece o no**:

$$
x_{ij} = \begin{cases} 1 & \text{si } t_j \text{ aparece al menos una vez en } d_i \\ 0 & \text{en otro caso} \end{cases}
$$

A diferencia de las otras dos representaciones del plan:

| Representación | Valor de $x_{ij}$ | Qué captura |
|---|---|---|
| Frecuencia | número de veces que $t_j$ aparece en $d_i$ | presencia + repetición |
| **Binaria** | **1 si aparece, 0 si no** | **solo presencia** |
| TF-IDF | frecuencia × rareza del término en el corpus | presencia + repetición + poder discriminante |

## Decisiones de diseño

| Decisión | Valor | Motivo |
|---|---|---|
| Clase | `CountVectorizer(binary=True)` | Lo pide el plan (sección Fase 3). Con `binary=True` todo conteo > 0 se convierte en 1 |
| Texto de entrada | `title_norm` y `abstract_norm` | Igual que TF-IDF: se ajusta sobre el texto ya normalizado |
| `lowercase` | `False` | La normalización ya pasó todo a minúsculas |
| `token_pattern` | `r"\S+"` | Mismo motivo que en TF-IDF: los tokens ya vienen separados por espacios y no se deben perder tokens de un carácter |
| `stop_words` | `None` | Ya se quitaron en la normalización |
| `ngram_range` | `(1, 1)` unigramas, `(2, 2)` bigramas | Lo pide el plan; bigramas formados tras quitar stopwords |
| `min_df` / `max_df` | Por defecto (`1` y `1.0`) | Mismo vocabulario que TF-IDF, así las 3 representaciones son comparables término a término |
| `dtype` | `int64` (por defecto) | Con `int8` el producto `X @ q.T` se desbordaría al sumar más de 127 coincidencias |
| Normalización | **No** se guarda normalizada | La matriz guardada es la binaria pura (0/1). En la Fase 4 la similitud coseno debe dividir entre las normas (ver sección 8) |
| Formato guardado | `pickle` con vectorizador, matriz CSR y `doc_ids` | Mismas llaves que los `.pkl` de TF-IDF, para que `retrieve()` cargue cualquier tipo igual |

## 1. Configuración

Se importan las librerías y se definen las rutas. `FIELDS` y `NGRAMS` son los mismos diccionarios del notebook 03; solo cambia `REPR = "binary"`, que va en el nombre de cada archivo.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NORM_PATH = ROOT / "data" / "scientific_normalized_corpus.tsv"
MODELS_DIR = ROOT / "models"
REPORT_PATH = ROOT / "results" / "binary_stats.md"
MODELS_DIR.mkdir(exist_ok=True)

# Campo del CLI -> columna normalizada del TSV
FIELDS = {"title": "title_norm", "abstract": "abstract_norm"}
# Tipo de n-grama del CLI -> ngram_range de scikit-learn
NGRAMS = {"unigram": (1, 1), "bigram": (2, 2)}
REPR = "binary"

print("scikit-learn", sklearn.__version__)

scikit-learn 1.9.1


## 2. Carga del corpus normalizado

Se lee el TSV como texto (`dtype=str`) y con `keep_default_na=False` para que un campo vacío quede como `""` y no como `NaN` (el vectorizador fallaría con `NaN`). La lista `doc_ids` guarda el `arxiv_id` de cada fila: la fila *i* de cada matriz corresponde a `doc_ids[i]`.

In [2]:
corpus = pd.read_csv(NORM_PATH, sep="\t", dtype=str, keep_default_na=False)
assert corpus["arxiv_id"].is_unique
doc_ids = corpus["arxiv_id"].tolist()

for col in FIELDS.values():
    empty = (corpus[col].str.strip() == "").sum()
    print(f"{col}: {len(corpus)} documentos, {empty} vacíos")

corpus[["arxiv_id", "title_norm"]].head(3)

title_norm: 500 documentos, 0 vacíos
abstract_norm: 500 documentos, 0 vacíos


,arxiv_id,title_norm
0,2610.00840,contextual trajectory incremental contextual d...
1,2610.00833,veritygate four gate schema level faithfulness...
2,2610.00827,verbalize internal probabilities couple large ...


## 3. Ejemplo pequeño: frecuencia vs binaria

Antes de usar el corpus real, se ve el efecto de `binary=True` con 3 documentos de juguete (ya "normalizados"). El mismo `CountVectorizer` se ajusta dos veces: una contando apariciones y otra solo marcando presencia.

In [3]:
toy = [
    "model learn model representation",
    "text model text text classification",
    "image classification",
]

params = dict(lowercase=False, token_pattern=r"\S+")
count_vec = CountVectorizer(**params).fit(toy)
binary_vec = CountVectorizer(**params, binary=True).fit(toy)

terms = count_vec.get_feature_names_out()
docs = ["doc0", "doc1", "doc2"]
pd.concat(
    {
        "frecuencia": pd.DataFrame(count_vec.transform(toy).toarray(), index=docs, columns=terms),
        "binaria": pd.DataFrame(binary_vec.transform(toy).toarray(), index=docs, columns=terms),
    },
    axis=1,
)

frecuencia                                              binaria  \
     classification image learn model representation text classification   
doc0              0     0     1     2              1    0              0   
doc1              1     0     0     1              0    3              1   
doc2              1     1     0     0              0    0              1   

                                            
     image learn model representation text  
doc0     0     1     1              1    0  
doc1     0     0     1              0    1  
doc2     1     0     0              0    0

Qué se observa:

- **Vocabulario:** es idéntico en ambos casos (orden alfabético). `binary=True` no cambia qué términos existen, solo el valor de cada celda.
- **`doc0`:** `model` aparece 2 veces → frecuencia 2, binaria 1.
- **`doc1`:** `text` aparece 3 veces → frecuencia 3, binaria 1.
- **Consecuencia:** en la representación binaria un término repetido pesa lo mismo que uno mencionado una sola vez. Cada documento se reduce al **conjunto** de sus términos, y la suma de una fila es el número de términos **distintos** del documento.

## 4. Vectorizador binario

`build_vectorizer` es la única fábrica de vectorizadores, para que las 4 configuraciones usen exactamente los mismos parámetros salvo `ngram_range`. Los parámetros de tokenización son los mismos que en `TfidfVectorizer` del notebook 03, por lo que el vocabulario de cada configuración binaria debe coincidir con el de TF-IDF (se verifica en la sección 7).

In [4]:
def build_vectorizer(ngram_range) -> CountVectorizer:
    return CountVectorizer(
        lowercase=False,       # el texto ya está en minúsculas
        token_pattern=r"\S+",  # los tokens ya vienen separados por espacios
        ngram_range=ngram_range,
        binary=True,           # 1 si el término aparece, 0 si no
    )


def model_path(field: str, ngram: str, repr_: str = REPR) -> Path:
    return MODELS_DIR / f"{field}_{ngram}_{repr_}.pkl"

## 5. Paso a paso con una configuración (abstract · unigrama)

Antes del ciclo de las 4 configuraciones, se desglosa lo que hace `fit_transform` sobre una sola.

### 5.1 `fit`: construir el vocabulario

`fit` recorre todos los abstracts, los parte en tokens con `token_pattern` y asigna a cada término distinto un índice de columna. El resultado queda en `vocabulary_` (término → columna).

In [5]:
demo_vec = build_vectorizer(NGRAMS["unigram"])
demo_vec.fit(corpus["abstract_norm"])

demo_terms = demo_vec.get_feature_names_out()
print("Tamaño del vocabulario:", len(demo_terms))
print("Primeros términos:", list(demo_terms[:10]))
print("Columna de 'transformer':", demo_vec.vocabulary_.get("transformer"))

Tamaño del vocabulario: 7958
Primeros términos: ['$', "'s", '+', '+0.0111', '+0.013', '+0.015', '+0.019', '+0.054', '+0.06', '+0.0602']
Columna de 'transformer': 7340


### 5.2 `transform`: construir la matriz documento-término

`transform` cuenta cada término de cada documento y, por `binary=True`, reemplaza todo conteo > 0 por 1. La salida es una matriz **dispersa** (`scipy.sparse`): solo se guardan las celdas distintas de cero, porque la gran mayoría son 0. Se convierte a CSR (`tocsr()`) para que extraer filas y multiplicar sea rápido.

In [6]:
X_demo = demo_vec.transform(corpus["abstract_norm"]).tocsr()

print("Forma (documentos, términos):", X_demo.shape)
print("Celdas distintas de cero:", X_demo.nnz)
print("Valores distintos guardados:", np.unique(X_demo.data))
print("Tipo de dato:", X_demo.dtype)

Forma (documentos, términos): (500, 7958)
Celdas distintas de cero: 52894
Valores distintos guardados: [1]
Tipo de dato: int64


### 5.3 Leer una fila

La fila 0 es el vector del primer documento. En formato CSR, `indices` son las columnas con 1 y `data` sus valores (todos 1). Al traducir las columnas con el vocabulario se recupera el **conjunto** de términos del abstract. Se compara contra el número de términos distintos del texto normalizado para confirmar que coincide.

In [7]:
row = X_demo[0]
row_terms = demo_terms[row.indices]

print(corpus.loc[0, "arxiv_id"], "|", corpus.loc[0, "title"])
print("Términos presentes (unos en la fila):", row.nnz)
print("Términos distintos en abstract_norm:", len(set(corpus.loc[0, "abstract_norm"].split())))
print("Valores de la fila:", np.unique(row.data))
print("Muestra de términos:", sorted(row_terms)[:15])

2610.00840 | Contextual trajectory and incremental contextual displacement: Towards using LLMs to understand dynamic, utterance-specific meaning construction
Términos presentes (unos en la fila): 103
Términos distintos en abstract_norm: 103
Valores de la fila: [1]
Muestra de términos: ["'s", 'add', 'also', 'alter', 'ambiguity', 'ambiguity-', 'analysis', 'approach', 'associate', 'base', 'case', 'characteristic', 'cls', 'construct', 'construction']


## 6. Ajuste y guardado de las 4 configuraciones

Para cada combinación campo × n-grama:

1. Se crea un vectorizador nuevo con `build_vectorizer`.
2. `fit_transform` aprende el vocabulario y genera la matriz binaria en un solo paso.
3. Se guarda un diccionario con el vectorizador (necesario para transformar consultas en la Fase 4), la matriz y `doc_ids`.
4. Se calculan las estadísticas que pide el plan: tamaño del vocabulario y dispersión.

En una matriz binaria `nnz` (número de unos) es exactamente el total de pares (documento, término distinto), así que `terminos_por_doc` es el promedio de términos distintos por documento.

In [8]:
rows = []
for field, col in FIELDS.items():
    for ngram, ngram_range in NGRAMS.items():
        vectorizer = build_vectorizer(ngram_range)
        matrix = vectorizer.fit_transform(corpus[col]).tocsr()

        payload = {
            "field": field,
            "ngram": ngram,
            "repr": REPR,
            "column": col,
            "vectorizer": vectorizer,
            "matrix": matrix,
            "doc_ids": doc_ids,
        }
        path = model_path(field, ngram)
        with open(path, "wb") as f:
            pickle.dump(payload, f)

        n_docs, n_terms = matrix.shape
        empty_rows = int((matrix.getnnz(axis=1) == 0).sum())
        rows.append({
            "config": path.stem,
            "documentos": n_docs,
            "vocabulario": n_terms,
            "no_ceros": matrix.nnz,
            "densidad_%": round(100 * matrix.nnz / (n_docs * n_terms), 3),
            "dispersion_%": round(100 * (1 - matrix.nnz / (n_docs * n_terms)), 3),
            "terminos_por_doc": round(matrix.nnz / n_docs, 1),
            "docs_sin_terminos": empty_rows,
        })
        print("Guardado", path.relative_to(ROOT))

stats = pd.DataFrame(rows).set_index("config")
stats

Guardado models/title_unigram_binary.pkl
Guardado models/title_bigram_binary.pkl
Guardado models/abstract_unigram_binary.pkl


Guardado models/abstract_bigram_binary.pkl


,documentos,vocabulario,no_ceros,densidad_%,dispersion_%,terminos_por_doc,docs_sin_terminos
config,,,,,,,
title_unigram_binary,500,1863,4262,0.458,99.542,8.5,0
title_bigram_binary,500,3545,3830,0.216,99.784,7.7,0
abstract_unigram_binary,500,7958,52894,1.329,98.671,105.8,0
abstract_bigram_binary,500,60073,69949,0.233,99.767,139.9,0


## 7. Verificaciones

Para cada `.pkl` guardado:

1. Se recarga y tiene la forma y los `doc_ids` correctos.
2. **Es binaria:** todos los valores guardados son 1.
3. **Coincide con la frecuencia:** la matriz binaria es igual a `(conteo > 0)` de un `CountVectorizer` sin `binary`.
4. **Mismo vocabulario que TF-IDF:** el vocabulario es idéntico al del `.pkl` TF-IDF correspondiente, y los unos están en las mismas celdas que los valores no nulos de TF-IDF. Así las 3 representaciones se comparan sobre los mismos términos.
5. Transformar de nuevo el corpus con el vectorizador guardado da la misma matriz (es lo que se hará con las consultas en la Fase 4).

In [9]:
for field, col in FIELDS.items():
    for ngram, ngram_range in NGRAMS.items():
        with open(model_path(field, ngram), "rb") as f:
            m = pickle.load(f)
        X = m["matrix"]

        # 1. forma y doc_ids
        assert m["doc_ids"] == doc_ids
        assert X.shape == (len(corpus), len(m["vectorizer"].vocabulary_))

        # 2. solo unos
        assert (X.data == 1).all()

        # 3. binaria == (frecuencia > 0)
        counts = CountVectorizer(lowercase=False, token_pattern=r"\S+",
                                 ngram_range=ngram_range).fit_transform(corpus[col])
        assert (X != (counts > 0)).nnz == 0

        # 4. mismo vocabulario y mismo patrón de no ceros que TF-IDF
        tfidf_path = model_path(field, ngram, "tfidf")
        if tfidf_path.exists():
            with open(tfidf_path, "rb") as f:
                t = pickle.load(f)
            assert t["vectorizer"].vocabulary_ == m["vectorizer"].vocabulary_
            assert (X != (t["matrix"] > 0)).nnz == 0
        else:
            print("Aviso: no existe", tfidf_path.name, "(ejecutar el notebook 03)")

        # 5. transformar de nuevo reproduce la matriz
        X2 = m["vectorizer"].transform(corpus[col])
        assert (X != X2).nnz == 0
print("Verificaciones correctas")

Verificaciones correctas


## 8. Similitud coseno con vectores binarios

La matriz binaria se guarda **sin normalizar**, por lo que el producto punto `X @ q.T` no es todavía el coseno. Para vectores 0/1:

- El producto punto $d \cdot q$ es el número de términos que comparten documento y consulta, $|D \cap Q|$.
- La norma $\lVert d \rVert = \sqrt{|D|}$, la raíz del número de términos distintos del documento.

$$
\cos(d, q) = \frac{|D \cap Q|}{\sqrt{|D|}\,\sqrt{|Q|}}
$$

Es decir, cuenta coincidencias y penaliza los documentos largos (con muchos términos distintos). En la Fase 4 basta con normalizar filas con `sklearn.preprocessing.normalize` (o usar `cosine_similarity`) antes del producto.

### 8.1 Comprobación de la fórmula

Se calcula el coseno de dos formas, con `normalize` y con la fórmula de conjuntos, y deben coincidir.

In [10]:
def load_model(field: str, ngram: str, repr_: str = REPR) -> dict:
    with open(model_path(field, ngram, repr_), "rb") as f:
        return pickle.load(f)


def cosine_scores(model: dict, query_norm: str) -> np.ndarray:
    """Coseno entre la consulta (ya normalizada) y cada documento."""
    q = model["vectorizer"].transform([query_norm])
    return (normalize(model["matrix"]) @ normalize(q).T).toarray().ravel()


m = load_model("abstract", "unigram")
query_norm = "misinformation detection use transformer"  # ya normalizada a mano
q = m["vectorizer"].transform([query_norm])
print("Términos de la consulta en el vocabulario:", q.nnz)

scores = cosine_scores(m, query_norm)

# Fórmula de conjuntos: |D ∩ Q| / (sqrt(|D|) * sqrt(|Q|))
shared = (m["matrix"] @ q.T).toarray().ravel()
doc_sizes = m["matrix"].getnnz(axis=1)
by_sets = shared / (np.sqrt(doc_sizes) * np.sqrt(q.nnz))
assert np.allclose(scores, by_sets)
print("normalize() y la fórmula de conjuntos coinciden")

Términos de la consulta en el vocabulario: 3
normalize() y la fórmula de conjuntos coinciden


### 8.2 Top-10 con la representación binaria

Orden descendente por puntaje y desempate estable por `arxiv_id` (decisión 4 del plan). La columna `compartidos` muestra $|D \cap Q|$: cuántos términos de la consulta tiene cada documento.

In [11]:
top = pd.DataFrame({
    "arxiv_id": doc_ids,
    "score": scores.round(4),
    "compartidos": shared,
    "terminos_doc": doc_sizes,
    "title": corpus["title"],
})
top = top.sort_values(["score", "arxiv_id"], ascending=[False, True], kind="stable").head(10)
top.insert(0, "rank", range(1, 11))
top_binary = top.reset_index(drop=True)
top_binary

,rank,arxiv_id,score,compartidos,terminos_doc,title
0,1,2609.39385,0.1868,3,86,"TTLab at Daleel 2026: STAR-Ar, Sequence Taggin..."
1,2,2610.00564,0.1333,2,75,Attention Kernels for Learning Maps Between He...
2,3,2609.39662,0.1291,2,80,Typographic Attack Against VLM-based AI-genera...
3,4,2610.10556,0.1283,2,81,LIFT: A Lifecycle-aware Interaction Factorizat...
4,5,2609.25306,0.1275,2,82,ReFilter: Bridging Embeddings and LLM Filterin...
5,6,2609.39585,0.1252,2,85,A Generalizable and Explainable Framework for ...
6,7,2610.00417,0.1252,2,85,Source Identification Is Not Fitness Testing: ...
7,8,2609.34802,0.1185,2,95,"No Attention, No Problem: Rethinking Session-b..."
8,9,2609.39184,0.1179,2,96,Fiber-Resolved Microstructure Quantification f...
9,10,2609.39635,0.1179,2,96,SAGE: Salient Factor Discovery and Generation ...


### 8.3 Comparación con TF-IDF

Misma consulta y misma configuración (abstract · unigrama) con el modelo TF-IDF del notebook 03. Las diferencias en el ranking vienen de que la binaria ignora cuántas veces aparece cada término y no da más peso a los términos raros (por ejemplo, `misinformation` vale lo mismo que `use`).

In [12]:
tfidf_path = model_path("abstract", "unigram", "tfidf")
if tfidf_path.exists():
    t = load_model("abstract", "unigram", "tfidf")
    t_scores = (t["matrix"] @ t["vectorizer"].transform([query_norm]).T).toarray().ravel()
    t_top = (
        pd.DataFrame({"arxiv_id": doc_ids, "score": t_scores})
        .sort_values(["score", "arxiv_id"], ascending=[False, True], kind="stable")
        .head(10)["arxiv_id"].tolist()
    )
    comparison = pd.DataFrame({
        "rank": range(1, 11),
        "binaria": top_binary["arxiv_id"],
        "tfidf": t_top,
    })
    common = len(set(t_top) & set(top_binary["arxiv_id"]))
    print(f"Documentos en común en el top-10: {common}/10")
    display(comparison)
else:
    print("No existe", tfidf_path.name, "(ejecutar el notebook 03)")

Documentos en común en el top-10: 6/10


,rank,binaria,tfidf
0,1,2609.39385,2609.39585
1,2,2610.00564,2609.39662
2,3,2609.39662,2609.39427
3,4,2610.10556,2609.39385
4,5,2609.25306,2610.00564
5,6,2609.39585,2609.39184
6,7,2610.00417,2609.39429
7,8,2609.34802,2609.39807
8,9,2609.39184,2609.39627
9,10,2609.39635,2610.00417


## 9. Reporte de la representación binaria

Se guarda en `results/binary_stats.md`. El vocabulario de cada configuración es el mismo que en `results/tfidf_stats.md`; lo que se reporta aquí confirma tamaño y dispersión de las matrices binarias.

In [13]:
from datetime import datetime, timezone

report = f"""# Representación binaria

- Generado: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}
- scikit-learn {sklearn.__version__}, `CountVectorizer(binary=True)`
- Parámetros: `lowercase=False`, `token_pattern=r"\\S+"`, `binary=True`, sin `min_df`/`max_df`
- Matriz guardada sin normalizar (valores 0/1); la similitud coseno normaliza filas en la Fase 4
- Entrada: `{NORM_PATH.relative_to(ROOT)}` ({len(corpus)} documentos)
- Modelos: `models/{{campo}}_{{ngrama}}_binary.pkl`

## Vocabulario y dispersión
{stats.to_markdown()}
"""
REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)

/home/mrangello/Documentos/PracticaML/results/binary_stats.md


## 10. Resultado

Archivos generados:
- `models/title_unigram_binary.pkl`, `models/title_bigram_binary.pkl`, `models/abstract_unigram_binary.pkl`, `models/abstract_bigram_binary.pkl`: cada uno con `vectorizer`, `matrix` (CSR, valores 0/1, sin normalizar) y `doc_ids`.
- `results/binary_stats.md`: tamaño de vocabulario y dispersión de cada matriz.

**Para la Fase 4:** con la representación binaria hay que normalizar las filas antes del producto punto (`normalize(X) @ normalize(q).T`), a diferencia de TF-IDF, cuya matriz ya viene con norma L2 = 1.

**Siguiente paso:** configuración de frecuencia (`CountVectorizer` sin `binary`) y Fase 4, recuperación con similitud coseno.